# Loaded the dataset

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd
from ml.src.data.load_data import load_raw


df = load_raw("D:\\HealthForecastAI\\ml\\data\\raw\\diabetic_data.csv")

df.head()

# EDA Analysis

In [ ]:
print("Dataset shape:", df.shape)


In [ ]:
print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
df.info()

# Checking null values 

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)

print(missing_percent)

# Check duplicate values

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Duplicate encounter IDs:", df["encounter_id"].duplicated().sum())

# Target variable analyze

In [ ]:
print("Readmission categories:")
print(df["readmitted"].value_counts())

In [ ]:
print("\nReadmission percentage:")
print(df["readmitted"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
df.describe().T

# Categorical columns identify


In [ ]:
categorical_cols = df.select_dtypes(include="object").columns

print("Number of categorical columns:", len(categorical_cols))
print("\nCategorical columns:")
print(categorical_cols.tolist())

# Distribution of categorical features

In [ ]:
categorical_analysis_cols = [
    "race",
    "gender",
    "age",
    "diabetesMed",
    "change"
]

for col in categorical_analysis_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

# Analyze the medications

In [ ]:
medication_cols = [
    "metformin",
    "insulin",
    "glipizide",
    "glyburide"
]

for col in medication_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

# Analyze diagnosis columns 

In [ ]:
diagnosis_cols = ["diag_1", "diag_2", "diag_3"]

for col in diagnosis_cols:
    print(f"{col}:")
    print("Unique values:", df[col].nunique(dropna=True))
    print()

In [ ]:
for col in diagnosis_cols:
    print(f"\n--- Top 10 {col} diagnoses ---")
    print(df[col].value_counts(dropna=True).head(10))

analysis — Hospital stay vs readmission

In [ ]:
df.groupby("readmitted")["time_in_hospital"].agg(
    ["count", "mean", "median"]
)

In [ ]:
df.groupby("readmitted")["number_emergency"].agg(
    ["count", "mean", "median", "max"]
)

In [ ]:
print(df["number_emergency"].describe())

In [ ]:
print(df.groupby("readmitted")["number_emergency"].mean())

In [ ]:
print("Are number_emergency and number_inpatient identical?")
print(df["number_emergency"].equals(df["number_inpatient"]))

In [ ]:
print(df[["number_inpatient", "number_emergency"]].head(20))

In [ ]:
print(
    df.groupby("readmitted")[["number_inpatient", "number_emergency"]].mean()
)

# Age group vs 30-day readmission

In [ ]:
age_readmission = pd.crosstab(
    df["age"],
    df["readmitted"],
    normalize="index"
) * 100

age_readmission.round(2)

# Visualization - Bar chart

In [ ]:
age_readmission["<30"].plot(
    kind="bar",
    figsize=(10, 5),
    title="30-Day Readmission Rate by Age Group",
    ylabel="Readmission Rate (%)",
    xlabel="Age Group"
)

plt.tight_layout()
plt.show()

 ### EDA Finding: Age and 30-Day Readmission

The 30-day readmission rate varies across age groups. The [20-30) age group has the highest observed 30-day readmission rate at 14.24%, while the [0-10) group has the lowest at 1.86%. Most older adult age groups show 30-day readmission rates around 10-12%.

These results indicate that age may contain useful information for readmission prediction, although the observed association does not imply causation.

In [ ]:
df.groupby("readmitted")["num_medications"].agg(
    ["count", "mean", "median", "max"]
)

In [ ]:
med_readmission = df.groupby("readmitted")["num_medications"].mean()

med_readmission.plot(
    kind="bar",
    figsize=(7, 5),
    title="Average Number of Medications by Readmission Status",
    ylabel="Average Number of Medications",
    xlabel="Readmission Status"
)

plt.tight_layout()
plt.show()

### EDA Finding: Medication Count and Readmission

The average number of medications is highest among patients readmitted within 30 days (16.90), followed by patients readmitted after 30 days (16.28), and patients with no readmission (15.67). This suggests a potential association between medication burden and readmission status, although this analysis does not establish causation.

# Initial cleaning

In [ ]:
from ml.src.data.preprocess import remove_expired_patients

In [ ]:
before = len(df)

df_no_expired = remove_expired_patients(df)

after = len(df_no_expired)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

In [ ]:
print(
    df_no_expired["discharge_disposition_id"]
    .isin([11, 19, 20, 21])
    .sum()
)

# Next step 
Connect the function to the basic ml pipeline 


In [ ]:
def basic_clean(frame, config):
    preprocessing = config.get("preprocessing", {})
    cleaned = drop_unused_columns(
        frame,
        preprocessing.get("drop_columns", [])
    )
    return cleaned.drop_duplicates()

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

In [ ]:
import pandas as pd

df = pd.read_csv("D:\\HealthForecastAI\\ml\\data\\raw\\diabetic_data.csv")

print(df.shape)

In [ ]:
from ml.src.data.preprocess import basic_clean

In [ ]:
cleaned_df = basic_clean(df, config)

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)
print("Remaining columns:", len(cleaned_df.columns))

In [ ]:
import yaml

with open("D:\\HealthForecastAI\\ml\\configs\\config.yaml", "r") as f:
    config = yaml.safe_load(f)

cleaned_df = basic_clean(df, config)

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)
print("Remaining columns:", len(cleaned_df.columns))

In [ ]:
print(len(df_no_expired))
print(len(cleaned_df))

# Initial cleaning completed 
Expired-patient records removed — 1,652 rows

Unnecessary/high-missing columns droped — 5 columns

to  remoce duplicate rows it is in the basic clean()

# Preprocessing started

In [ ]:
# Convert readmission target into binary format

df_processed = cleaned_df.copy()

df_processed["readmitted_binary"] = (
    df_processed["readmitted"].astype(str).str.strip() == "<30"
).astype(int)

print(df_processed["readmitted_binary"].value_counts())
print("\nTarget percentage:")
print(df_processed["readmitted_binary"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Replace original target with binary target

df_processed = df_processed.drop(columns=["readmitted"])

df_processed = df_processed.rename(
    columns={"readmitted_binary": "readmitted"}
)

print(df_processed["readmitted"].value_counts())
print(df_processed.shape)

# checking missing values

In [ ]:
# Check missing values in the processed dataset

missing = df_processed.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

print("\nTotal missing values:", df_processed.isnull().sum().sum())

In [ ]:
print("--- max_glu_serum ---")
print(df_processed["max_glu_serum"].value_counts(dropna=False))

print("\n--- A1Cresult ---")
print(df_processed["A1Cresult"].value_counts(dropna=False))

# identifgy most frequent values

In [ ]:
# Check the most frequent value for categorical missing columns

for column in ["max_glu_serum", "A1Cresult"]:
    print(f"{column}: {df_processed[column].mode()[0]}")

In [ ]:
# Impute missing categorical values using the most frequent value

for column in ["max_glu_serum", "A1Cresult"]:
    df_processed[column] = df_processed[column].fillna(
        df_processed[column].mode()[0]
    )

print("Remaining missing values:")
print(df_processed.isnull().sum()[df_processed.isnull().sum() > 0])

In [ ]:
# Identify numeric and categorical features

numeric_columns = df_processed.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = df_processed.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Remove target from feature columns
numeric_columns.remove("readmitted")

print("Numeric columns:", len(numeric_columns))
print(numeric_columns)

print("\nCategorical columns:", len(categorical_columns))
print(categorical_columns)

In [ ]:
# Check frequency of diagnosis codes

for column in ["diag_1", "diag_2", "diag_3"]:
    print(f"\n{column}")
    print("Unique values:", df_processed[column].nunique())
    print("Rare values (appearing <= 10 times):",
          (df_processed[column].value_counts() <= 10).sum())

In [ ]:
# Percentage of records containing rare diagnosis codes

for column in ["diag_1", "diag_2", "diag_3"]:
    counts = df_processed[column].value_counts()
    rare_codes = counts[counts <= 10].index

    rare_mask = df_processed[column].isin(rare_codes)

    print(f"\n{column}")
    print("Rows with rare codes:", rare_mask.sum())
    print("Percentage:", round(rare_mask.mean() * 100, 2), "%")

In [ ]:
# Collapse rare diagnosis codes into "Other"

diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    counts = df_processed[column].value_counts()
    rare_codes = counts[counts <= 10].index

    df_processed[column] = df_processed[column].replace(
        rare_codes, "Other"
    )

# Verify the result
for column in diagnosis_columns:
    print(f"\n{column}")
    print("Unique values after grouping:", df_processed[column].nunique())
    print(df_processed[column].value_counts().tail())

In [ ]:
# Check for '?' missing-value tokens

question_mark_counts = {}

for column in categorical_columns:
    count = (df_processed[column] == "?").sum()
    if count > 0:
        question_mark_counts[column] = count

print("Columns containing '?' token:")
print(question_mark_counts)

In [ ]:
question_mark_counts = {}

for column in categorical_columns:
    count = (df_processed[column] == "?").sum()
    if count > 0:
        question_mark_counts[column] = count

print("Columns containing '?' token:")
print(question_mark_counts)

In [ ]:
# Convert the configured missing-value token '?' into NaN

df_processed = df_processed.replace("?", pd.NA)

# Check again
question_mark_counts = {}

for column in categorical_columns:
    count = (df_processed[column] == "?").sum()
    if count > 0:
        question_mark_counts[column] = count

print("Remaining '?' values:")
print(question_mark_counts)

In [ ]:
# Check missing values after converting '?' to NaN

missing = df_processed.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

print("\nTotal missing values:", df_processed.isna().sum().sum())

In [ ]:
# Impute categorical missing values using the most frequent value

for column in categorical_columns:
    if df_processed[column].isna().any():
        df_processed[column] = df_processed[column].fillna(
            df_processed[column].mode()[0]
        )

# Verify
missing = df_processed.isna().sum()
missing = missing[missing > 0]

print("Remaining missing values:")
print(missing)

print("\nTotal missing values:", df_processed.isna().sum().sum())

# One hot encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Create One-Hot Encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit and transform categorical features
encoded_categorical = encoder.fit_transform(
    df_processed[categorical_columns]
)

print("Original categorical features:", len(categorical_columns))
print("Encoded feature count:", encoded_categorical.shape[1])

In [ ]:
# Get the names of the encoded categorical features

encoded_feature_names = encoder.get_feature_names_out(categorical_columns)

print("Number of encoded feature names:", len(encoded_feature_names))
print("\nFirst 20 encoded features:")
print(encoded_feature_names[:20])

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale numeric features
scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(
    df_processed[numeric_columns]
)

print("Original numeric features:", len(numeric_columns))
print("Scaled numeric shape:", scaled_numeric.shape)

In [ ]:
import numpy as np

# Combine scaled numeric and encoded categorical features
X_processed = np.hstack([
    scaled_numeric,
    encoded_categorical
])

print("Final feature matrix shape:", X_processed.shape)

In [ ]:
# Create final processed dataframe with feature names

numeric_feature_names = numeric_columns

all_feature_names = (
    numeric_feature_names + encoded_feature_names.tolist()
)

X_final = pd.DataFrame(
    X_processed,
    columns=all_feature_names,
    index=df_processed.index
)

y_final = df_processed["readmitted"].astype(int)

print("X shape:", X_final.shape)
print("y shape:", y_final.shape)

print("\nTarget distribution:")
print(y_final.value_counts())

In [ ]:
# Combine features and target into final processed dataset

processed_df = X_final.copy()
processed_df["readmitted"] = y_final

output_path = r"D:\HealthForecastAI\ml\data\processed\admissions_features.parquet"

processed_df.to_parquet(output_path, index=False)

print("Processed dataset saved successfully.")
print("Shape:", processed_df.shape)
print("Path:", output_path)

In [ ]:
# Verify the saved processed dataset

check_df = pd.read_parquet(output_path)

print("Loaded shape:", check_df.shape)
print("Missing values:", check_df.isna().sum().sum())
print("Target distribution:")
print(check_df["readmitted"].value_counts())

In [ ]:
import importlib
import ml.src.data.preprocess as preprocess

importlib.reload(preprocess)

basic_clean = preprocess.basic_clean
build_preprocessor = preprocess.build_preprocessor

print("Preprocess module reloaded successfully!")

In [ ]:
cleaned_test = basic_clean(df, config)

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_test.shape)
print("Missing values:", cleaned_test.isna().sum().sum())

In [ ]:
print("readmitted present:", "readmitted" in cleaned_test.columns)
print("Columns:", cleaned_test.columns.tolist())

In [ ]:
import importlib
import ml.src.data.preprocess as preprocess

importlib.reload(preprocess)

basic_clean = preprocess.basic_clean
build_preprocessor = preprocess.build_preprocessor

print("Preprocess module reloaded successfully!")

In [ ]:
# Separate features and target
X_test = cleaned_test.drop(columns=["readmitted"])
y_test = (cleaned_test["readmitted"] == "<30").astype(int)

# Build the reusable preprocessing pipeline
preprocessor = build_preprocessor(cleaned_test, config)

# Fit and transform features
X_transformed = preprocessor.fit_transform(X_test)

print("Transformed shape:", X_transformed.shape)
print("Target shape:", y_test.shape)

print("\nTarget distribution:")
print(y_test.value_counts())

In [ ]:
print("Any NaN in transformed data:", pd.isna(X_transformed).sum())